In [15]:
import math
import gurobipy as gp
from gurobipy import GRB

items = list(range(20))
weights = [4, 1, 1, 2, 10, 2, 3, 9, 2, 5, 8, 7, 9, 6, 6, 11, 9, 7, 13, 4]
assert len(weights) == 20

people = ["F", "G", "H"]  # Fred, Ginger, Heather
B = 50  # capacity per backpack

def h_true(x):
    return 50 - 45 * math.atan((math.pi * x - 25) / 50.0)

# 10-segment secant-line PWL on x in [0, 50], segments of length 5
segments = list(range(1, 11))  # k = 1..10
seg_len = 5.0
alpha = {
    1: -2.519127, 2: -2.802291, 3: -2.600638, 4: -2.080390, 5: -1.546538,
    6: -1.132696, 7: -0.840751, 8: -0.638504, 9: -0.496778, 10: -0.395294,
}
beta = {
    1: 70.864142, 2: 72.279961, 3: 70.263427, 4: 62.459709, 5: 51.782663,
    6: 41.436613, 7: 32.678264, 8: 25.599631, 9: 19.930593, 10: 15.363823,
}

m = gp.Model("a3q2")
y = m.addVars(items, people, vtype=GRB.BINARY, name="y")
x = m.addVars(people, lb=0.0, name="x")

H_F = m.addVar(lb=-GRB.INFINITY, name="H_F")
H_G = m.addVar(lb=-GRB.INFINITY, name="H_G")
H_H = m.addVar(lb=-GRB.INFINITY, name="H_H")

z = m.addVars(segments, vtype=GRB.BINARY, name="z")
t = m.addVars(segments, lb=0.0, name="t")

# Each item assigned to exactly one person
m.addConstrs((gp.quicksum(y[i, p] for p in people) == 1 for i in items), name="assign_once")

# Link carried weights
m.addConstr(x["F"] == gp.quicksum(weights[i] * y[i, "F"] for i in items), name="xF_link")
m.addConstr(x["G"] == gp.quicksum(weights[i] * y[i, "G"] for i in items), name="xG_link")
m.addConstr(x["H"] == gp.quicksum(weights[i] * y[i, "H"] for i in items), name="xH_link")

# Capacity constraints
m.addConstr(x["F"] <= B, name="cap_F")
m.addConstr(x["G"] <= B, name="cap_G")
m.addConstr(x["H"] <= B, name="cap_H")

# Fred & Ginger happiness
m.addConstr(H_F == 100 - 1.7 * x["F"], name="H_F_def")
m.addConstr(H_G == 90 - 1.8 * x["G"], name="H_G_def")

# Heather PWL activation
m.addConstr(gp.quicksum(z[k] for k in segments) == 1, name="one_segment")
m.addConstrs((t[k] <= seg_len * z[k] for k in segments), name="t_bounds")

# x_H equals segment start + local offset
m.addConstr(
    x["H"] == gp.quicksum((5.0*(k-1))*z[k] + t[k] for k in segments),
    name="xH_PWL_position"
)

# H_H equals the segment's linear expression
m.addConstr(
    H_H == gp.quicksum(alpha[k]*((5.0*(k-1))*z[k] + t[k]) + beta[k]*z[k] for k in segments),
    name="H_H_PWL_value"
)

# Objective: maximize total happiness
m.setObjective(H_F + H_G + H_H, GRB.MAXIMIZE)

m.optimize()

if m.status == GRB.OPTIMAL:
    assign = {p: [] for p in people}
    for i in items:
        for p in people:
            if y[i, p].X > 0.5:
                assign[p].append(i+1)  # 1-based label

    xF, xG, xH = x["F"].X, x["G"].X, x["H"].X
    HF, HG, HH = H_F.X, H_G.X, H_H.X
    total = HF + HG + HH

    h_true_val = h_true(xH)
    abs_err = abs(h_true_val - HH)

    active_k = [k for k in segments if z[k].X > 0.5]
    k_sel = active_k[0] if active_k else None
    t_sel = t[k_sel].X if k_sel is not None else None

    print("\n=== Optimal Solution ===")
    print(f"Objective (total happiness): {total:.6f}\n")
    print(f"Fred:    items={assign['F']}, weight={xF:.2f}, H_F={HF:.4f}")
    print(f"Ginger:  items={assign['G']}, weight={xG:.2f}, H_G={HG:.4f}")
    print(f"Heather: items={assign['H']}, weight={xH:.2f}, H_H(approx)={HH:.4f}\n")
    print(f"Heather true h(x_H) = {h_true_val:.4f}")
    print(f"Approx error        = {abs_err:.6f}")
    if k_sel is not None:
        seg_start = 5.0*(k_sel-1)
        print(f"Segment k           = {k_sel} (interval [{seg_start:.0f}, {seg_start+5:.0f}])")
        print(f"Local offset t_k    = {t_sel:.4f}  (x_H = {seg_start:.0f} + {t_sel:.4f} = {xH:.4f})")
    m.write("a3q2.lp")
else:
    print(f"Model status: {m.status} (not optimal)")


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 41 rows, 86 columns and 201 nonzeros
Model fingerprint: 0x3b89f19b
Variable types: 16 continuous, 70 integer (70 binary)
Coefficient statistics:
  Matrix range     [6e-02, 7e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Presolve removed 6 rows and 3 columns
Presolve time: 0.00s
Presolved: 35 rows, 83 columns, 173 nonzeros
Variable types: 10 continuous, 73 integer (70 binary)
Found heuristic solution: objective 61.9944170
Found heuristic solution: objective 62.1850050

Root relaxation: objective 6.639912e+01, 48 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0   66.3991

In [16]:
import pandas as pd
# for my screenshot - made this to print out nicely for legibility
data = [
    (1, "[0,5]",   -2.519127, 70.864142),
    (2, "[5,10]",  -2.802291, 72.279961),
    (3, "[10,15]", -2.600638, 70.263427),
    (4, "[15,20]", -2.080390, 62.459709),
    (5, "[20,25]", -1.546538, 51.782663),
    (6, "[25,30]", -1.132696, 41.436613),
    (7, "[30,35]", -0.840751, 32.678264),
    (8, "[35,40]", -0.638504, 25.599631),
    (9, "[40,45]", -0.496778, 19.930593),
    (10,"[45,50]", -0.395294, 15.363823),
]

df = pd.DataFrame(data, columns=["k", "Interval", "alpha_k", "beta_k"])
pd.set_option("display.float_format", "{:.6f}".format)
df

,k,Interval,alpha_k,beta_k
0,1,"[0,5]",-2.519127,70.864142
1,2,"[5,10]",-2.802291,72.279961
2,3,"[10,15]",-2.600638,70.263427
3,4,"[15,20]",-2.080390,62.459709
4,5,"[20,25]",-1.546538,51.782663
5,6,"[25,30]",-1.132696,41.436613
6,7,"[30,35]",-0.840751,32.678264
7,8,"[35,40]",-0.638504,25.599631
8,9,"[40,45]",-0.496778,19.930593
9,10,"[45,50]",-0.395294,15.363823
